In [2]:
!pip -q install --upgrade transformers datasets accelerate evaluate scikit-learn torch

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForMaskedLM, DataCollatorWithPadding,
                          DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments, set_seed)
from datasets import Dataset, DatasetDict
import evaluate

SEED = 42
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilroberta-base"
print("Device:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/

In [4]:
train_path = "data/train.csv"
test_path = "data/test.csv"
mlm_path = "data/mlm.txt"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train_df columns: ", train_df.columns.tolist())
print("test_df columns: ", test_df.columns.tolist())

len(train_df), len(test_df)

train_df.head()

train_df columns:  ['text', 'label']
test_df columns:  ['text', 'label']


,text,label
0,— Кровь! какую кровь? — встревожилась,1
1,– Под нижнюю подушку.,0
2,— Благодарю-с...,1
3,— Когда же это-с?,1
4,"Старуха помолчала, как бы в раздумье,",1


In [5]:

NORM_DASHES = r"[\u2012\u2013\u2014\u2015\-]+"  # разные тире/дефисы
NORM_QUOTES = {
    "“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
    "’": "'", "‘": "'", "‚": "'", "ʼ": "'", "′": "'"
}

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\u00a0", " ")  # NBSP
    # normalize quotes
    for k, v in NORM_QUOTES.items():
        s = s.replace(k, v)
    # normalize dashes
    s = re.sub(NORM_DASHES, " — ", s)
    # remove control chars except \n and \t
    s = re.sub(r"[\x00-\x09\x0b-\x1f\x7f]", " ", s)
    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

train_df["text"] = train_df["text"].astype(str).map(normalize_text)
test_df["text"]  = test_df["text"].astype(str).map(normalize_text)

train_df.head(3)

,text,label
0,— Кровь! какую кровь? — встревожилась,1
1,— Под нижнюю подушку.,0
2,— Благодарю — с...,1


In [6]:
train_split, valid_split = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"]
)
train_split = train_split.reset_index(drop=True)
valid_split = valid_split.reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_example(batch):
    return tokenizer(batch["text"], truncation=True, padding=False, max_length=256)

ds = DatasetDict({
    "train": Dataset.from_pandas(train_split[["text", "label"]], preserve_index=False),
    "val": Dataset.from_pandas(valid_split[["text", "label"]], preserve_index=False)
}).map(tokenize_example, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if device=="cuda" else None)

num_labels = 2
model_cls_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/4512 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

training_args_base = TrainingArguments(
    output_dir="out_cls_base",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    seed=SEED,
    report_to="none"
)

trainer_base = Trainer(
    model=model_cls_base,
    args=training_args_base,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result_base = trainer_base.train()
metrics_base_val = trainer_base.evaluate()

metrics_base_val

/tmp/ipython-input-2543182434.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_base = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.655332,0.624113,0.622643
2,No log,0.601378,0.671986,0.668264
3,No log,0.563310,0.693262,0.692271
4,0.620400,0.590456,0.678191,0.651709
5,0.620400,0.554214,0.719858,0.719451
6,0.620400,0.524253,0.723404,0.720795
7,0.620400,0.526107,0.735816,0.731290
8,0.506200,0.517477,0.735816,0.733127
9,0.506200,0.530611,0.742021,0.738153
10,0.506200,0.532718,0.737589,0.734064


{'eval_loss': 0.5306108593940735,
 'eval_accuracy': 0.7420212765957447,
 'eval_f1': 0.7381532860061408,
 'eval_runtime': 1.8546,
 'eval_samples_per_second': 608.205,
 'eval_steps_per_second': 19.411,
 'epoch': 10.0}

In [8]:
preds_base = trainer_base.predict(ds["val"])
y_true = preds_base.label_ids
y_pred = preds_base.predictions.argmax(axis=-1)
cm_base = confusion_matrix(y_true, y_pred)

print("Classification report:")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion matrix:")
print(cm_base)

Classification report:
              precision    recall  f1-score   support

           0     0.7854    0.6248    0.6959       533
           1     0.7159    0.8471    0.7760       595

    accuracy                         0.7420      1128
   macro avg     0.7506    0.7359    0.7360      1128
weighted avg     0.7487    0.7420    0.7382      1128

Confusion matrix:
[[333 200]
 [ 91 504]]


In [9]:
ds_test = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False).map(tokenize_example, batched=True)
preds_test = trainer_base.predict(ds_test.remove_columns(["label"]))
test_labels = preds_test.predictions.argmax(axis=-1)

submission = pd.DataFrame({"label": test_labels.astype(int)})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
submission.head()

Map:   0%|          | 0/1440 [00:00<?, ? examples/s]

,label
0,1
1,1
2,1
3,1
4,1


In [10]:
with open(mlm_path, "r", encoding="utf-8") as f:
    corpus = f.read()
corpus = corpus[:20000]
split_idx = int(0.9 * len(corpus))
corpus_train, corpus_val = corpus[:split_idx], corpus[split_idx:]

ds_mlm = DatasetDict({
    "train": Dataset.from_dict({"text": corpus_train}),
    "val": Dataset.from_dict({"text": corpus_val})
})

def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["text"]])

tokenized_ds_mlm = ds_mlm.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=ds_mlm["train"].column_names,
)

block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

lm_dataset = tokenized_ds_mlm.map(group_texts, batched=True, num_proc=4)

mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)

mlm_eval_args = TrainingArguments(
    output_dir="out_mlm_eval",
    per_device_eval_batch_size=64,
    dataloader_drop_last=False,
    report_to="none"
)
mlm_eval_trainer = Trainer(
    model=mlm_model,
    args=mlm_eval_args,
    eval_dataset=lm_dataset["val"],
    data_collator=mlm_collator,
)

eval_before = mlm_eval_trainer.evaluate()
loss_before = float(eval_before["eval_loss"])
print(f"MLM BEFORE — loss: {loss_before}")

Map (num_proc=4):   0%|          | 0/18000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/18000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


MLM BEFORE — loss: 3.7520554065704346


In [26]:
mlm_best_path = "my_awesome_mlm_model"

mlm_training_args = TrainingArguments(
    output_dir="my_awesome_mlm_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    logging_steps=50,
    metric_for_best_model="eval_loss",
    report_to="none"
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["val"],
    data_collator=mlm_collator,
    tokenizer=tokenizer,
)

mlm_trainer.train()

eval_after = mlm_trainer.evaluate()
loss_after = float(eval_after["eval_loss"])
print(f"MLM AFTER — loss: {loss_after}")

/tmp/ipython-input-2710116763.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  mlm_trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,1.879800,1.637497
2,1.726000,1.497300
3,1.592800,1.463048


MLM AFTER — loss: 1.4297361373901367


In [29]:
best_mlm_model = AutoModelForSequenceClassification.from_pretrained(
    "my_awesome_mlm_model/checkpoint-162", num_labels=num_labels, ignore_mismatched_sizes=True
).to(device)

training_args_base = TrainingArguments(
    output_dir="out_cls_mlm",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    seed=SEED,
    report_to="none"
)

trainer_base = Trainer(
    model=best_mlm_model,
    args=training_args_base,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result_base = trainer_base.train()
metrics_base_val = trainer_base.evaluate()

metrics_base_val

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at my_awesome_mlm_model/checkpoint-162 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3268878478.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_base = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.642430,0.620567,0.620488
2,No log,0.583867,0.664894,0.660097
3,No log,0.566517,0.690603,0.678230
4,0.609500,0.577079,0.682624,0.656692
5,0.609500,0.528260,0.713652,0.713692
6,0.609500,0.552612,0.715426,0.705721
7,0.609500,0.533427,0.716312,0.709134
8,0.488800,0.534453,0.726950,0.720041
9,0.488800,0.554822,0.727837,0.717367
10,0.488800,0.553667,0.727837,0.720194


{'eval_loss': 0.553666889667511,
 'eval_accuracy': 0.7278368794326241,
 'eval_f1': 0.7201942717925617,
 'eval_runtime': 0.7656,
 'eval_samples_per_second': 1473.435,
 'eval_steps_per_second': 47.025,
 'epoch': 10.0}

In [30]:
ds_test = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False).map(tokenize_example, batched=True)
preds_test = trainer_base.predict(ds_test.remove_columns(["label"]))
test_labels = preds_test.predictions.argmax(axis=-1)

submission = pd.DataFrame({"label": test_labels.astype(int)})
submission_path = "submission_mlm.csv"
submission.to_csv(submission_path, index=False)
submission.head()

Map:   0%|          | 0/1440 [00:00<?, ? examples/s]

,label
0,1
1,1
2,1
3,1
4,1
